# GalaxyMatch AI — Samsung Galaxy Shopping Assistant

Built by **Member 3 (UI)**. Runs on the real dataset (`data/phones.csv`,
15 phones) processed by `src/data_pipeline.py`, scored and ranked by
`src/recommender.py`, with personas/free-text extraction in
`src/personas.py`, badges in `src/badges.py`, and explanations in
`src/ai_assistant.py` (local Llama 3.2 3B via Ollama when available,
rule-based templates otherwise). Run `python -m src.data_pipeline` once
before this notebook if `data/normalized_scores.csv` doesn't exist yet —
the cell below falls back to computing it in-memory either way.


In [1]:
import sys, os, time, html as html_lib
sys.path.insert(0, os.path.abspath(".."))

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

from src import theme, data_pipeline, recommender, personas as personas_module, badges, ai_assistant


class _Engine:
    """Thin facade over the real modules so the rest of this notebook
    doesn't care which teammate's file a given function lives in."""
    mock_phones_df = staticmethod(data_pipeline.run_pipeline)
    recommend_phone = staticmethod(recommender.recommend_phone)
    rank_results = staticmethod(recommender.rank_results)
    recommend_all_personas = staticmethod(recommender.recommend_all_personas)
    compare_phones = staticmethod(recommender.compare_phones)
    assign_badges = staticmethod(badges.assign_badges)
    generate_explanation = staticmethod(ai_assistant.generate_explanation)
    extract_preferences_from_text = staticmethod(personas_module.extract_preferences_from_text)
    refine_preferences = staticmethod(personas_module.refine_preferences)


engine = _Engine()

theme.inject_theme()
phones_df = engine.mock_phones_df()
PERSONAS = personas_module.PERSONAS


In [2]:
display(HTML(f'''
<div class="gm-wrap" style="padding-bottom:8px;">
  <div class="gm-hero">
    <div class="gm-eyebrow">{theme.icon("star")} Galaxy AI Shopping Assistant</div>
    <h1 class="gm-title">Galaxy<span>Match</span> AI</h1>
    <p class="gm-tagline">Tell us who you are. We\'ll find your perfect Galaxy.</p>
  </div>
</div>
'''))


## Choose a Persona, or Describe Yourself

In [3]:
# --- Widget declarations -----------------------------------------------

persona_dropdown = widgets.Dropdown(
    options=[(p["name"], pid) for pid, p in PERSONAS.items()],
    value="arjun",
    description="Persona:",
)
persona_bio = widgets.HTML()
persona_panel = widgets.VBox([persona_dropdown, persona_bio])

describe_input = widgets.Textarea(
    placeholder="I'm a college student. Budget 30000. I play BGMI a lot and need good battery life.",
    layout=widgets.Layout(width="100%", height="90px"),
)
describe_hint = widgets.HTML(
    '<p style="font-size:12px;color:var(--gm-on-surface-variant);'
    'font-family:var(--gm-font);margin:6px 2px 0;">'
    "GalaxyMatch reads this and infers what matters most — camera, "
    "performance, battery, or value.</p>"
)
describe_panel = widgets.VBox([describe_input, describe_hint])

input_tabs = widgets.Tab(children=[persona_panel, describe_panel])
input_tabs.set_title(0, "Choose a Persona")
input_tabs.set_title(1, "Describe Yourself")

find_button = widgets.Button(description="Find My Galaxy →", button_style="primary")
find_button.add_class("gm-cta-btn")

thinking_output = widgets.Output()
results_output = widgets.Output()
compare_output = widgets.Output()

input_box = widgets.VBox([input_tabs, find_button, thinking_output])
input_box.add_class("gm-widget-panel")

# Shared state across callbacks (plain kernel-scope variables — no backend needed)
current_state = {"weights": None, "budget_min": None, "budget_max": None, "top3": None}


In [4]:
# --- Rendering + handlers ------------------------------------------------

SUB_SCORES = [
    ("camera_score", "Camera"),
    ("performance_score", "Performance"),
    ("battery_score", "Battery"),
    ("display_score", "Display"),
    ("value_score", "Value"),
]


def render_persona_bio(pid):
    p = PERSONAS[pid]
    persona_bio.value = f'''
    <div class="gm-persona-bio">
      <div class="gm-persona-avatar">{p["avatar"]}</div>
      <div>
        <p class="gm-persona-name">{p["name"]}</p>
        <p class="gm-persona-need">{html_lib.escape(p["need"])}</p>
        <p class="gm-persona-budget">₹{p["budget_min"]:,} – ₹{p["budget_max"]:,}</p>
      </div>
    </div>
    '''


def render_card(rank, row, badges_map):
    is_best = rank == 0
    phone_img = theme.phone_svg(row["model_name"])
    explanation = engine.generate_explanation(current_state["weights"], row)
    row_badges = badges_map.get(row["model_name"], [])
    badge_html = "".join(
        f'<span class="gm-tag-chip">{theme.icon(icon_name)} {label}</span>'
        for icon_name, label in row_badges
    )
    breakdown_html = "".join(
        f'<div class="gm-sub-row"><span class="gm-sub-label">{label}</span>'
        f'<div class="gm-sub-track"><div class="gm-sub-fill" '
        f'style="width:{row[col] * 10:.0f}%"></div></div>'
        f'<span class="gm-sub-val">{row[col]:.1f}</span></div>'
        for col, label in SUB_SCORES
    )
    top_chip = f'{theme.icon("star")} Best Match' if is_best else row["target_segment"].title()
    return f'''
    <div class="gm-card {"gm-best" if is_best else ""}">
      <div class="gm-ghost-rank">{rank + 1}</div>
      <div class="gm-card-top">
        <span class="gm-tag-chip">{top_chip}</span>
      </div>
      <div class="gm-phone-render">
        <div class="gm-phone-glow" style="background:rgba(188,195,255,0.45)"></div>
        {phone_img}
      </div>
      <p class="gm-phone-name">{row["model_name"]}</p>
      <p class="gm-explanation">{explanation}</p>
      <div class="gm-meter-label"><span>Match Score</span><span>{row["match_pct"]}%</span></div>
      <div class="gm-meter-track">
        <div class="gm-meter-fill" style="width:{row["match_pct"]}%;background:var(--gm-primary)">
          <div class="gm-edge"></div>
        </div>
      </div>
      {f'<div style="margin-top:10px;display:flex;gap:6px;flex-wrap:wrap;">{badge_html}</div>' if badge_html else ""}
      <details class="gm-breakdown"><summary>Why this score?</summary>{breakdown_html}</details>
      <div class="gm-spec-strip">
        <div><b>₹{row["price_inr"]:,}</b><span>Price</span></div>
        <div><b>{row["camera_mp"]}MP</b><span>Camera</span></div>
        <div><b>{row["battery_mah"]}mAh</b><span>Battery</span></div>
      </div>
    </div>
    '''


def render_results(weights, budget_min, budget_max):
    scored = engine.recommend_phone(weights, budget_min, budget_max, phones_df)
    top3 = engine.rank_results(scored, top_n=3)
    badges_map = engine.assign_badges(phones_df)
    current_state.update(weights=weights, budget_min=budget_min, budget_max=budget_max, top3=top3)

    cards_html = "".join(
        render_card(i, row, badges_map) for i, (_, row) in enumerate(top3.iterrows())
    )
    page_html = f'''
    <div class="gm-wrap" style="padding-top:8px;">
      <div class="gm-results-header">
        <h2>Your Top 3 Matches</h2>
        <p>Ranked by a transparent weighted-sum score — not a black box.</p>
      </div>
      <div class="gm-cards">{cards_html}</div>
    </div>
    '''
    with results_output:
        clear_output(wait=True)
        display(HTML(page_html))

    update_sliders(weights)
    rebuild_compare_controls(top3)


def on_persona_change(change):
    if change["name"] == "value":
        render_persona_bio(change["new"])


def on_find_click(_):
    with thinking_output:
        clear_output(wait=True)
        display(HTML(
            '<div class="gm-thinking">Matching your priorities '
            '<span class="gm-dots"><span></span><span></span><span></span></span></div>'
        ))
    time.sleep(0.4)  # brief pause so the loading state is visible, not just a flash

    if input_tabs.selected_index == 1:
        prefs = engine.extract_preferences_from_text(describe_input.value)
        weights, budget_min, budget_max = prefs["weights"], prefs["budget_min"], prefs["budget_max"]
    else:
        persona = PERSONAS[persona_dropdown.value]
        weights, budget_min, budget_max = persona["weights"], persona["budget_min"], persona["budget_max"]

    render_results(weights, budget_min, budget_max)
    with thinking_output:
        clear_output()


render_persona_bio(persona_dropdown.value)
persona_dropdown.observe(on_persona_change, names="value")
find_button.on_click(on_find_click)

display(input_box)
display(results_output)


Output()

## Premium & Insight Features

Five features that make the weighted-sum model's transparency *visible and
interactive* rather than just a prettier printout.


### Live "What-If" Weight Sliders

In [5]:
camera_slider = widgets.FloatSlider(value=0.25, min=0, max=1, step=0.05, description="Camera")
performance_slider = widgets.FloatSlider(value=0.25, min=0, max=1, step=0.05, description="Performance")
battery_slider = widgets.FloatSlider(value=0.25, min=0, max=1, step=0.05, description="Battery")
value_slider = widgets.FloatSlider(value=0.25, min=0, max=1, step=0.05, description="Value")
_sliders = [camera_slider, performance_slider, battery_slider, value_slider]
_slider_keys = ["camera", "performance", "battery", "value"]
_slider_sync_guard = {"active": False}


def update_sliders(weights):
    _slider_sync_guard["active"] = True
    for key, slider in zip(_slider_keys, _sliders):
        slider.value = round(weights[key], 2)
    _slider_sync_guard["active"] = False


def on_slider_change(_change):
    if _slider_sync_guard["active"] or current_state["budget_min"] is None:
        return
    raw = {k: s.value for k, s in zip(_slider_keys, _sliders)}
    total = sum(raw.values()) or 1.0
    normalized = {k: v / total for k, v in raw.items()}
    render_results(normalized, current_state["budget_min"], current_state["budget_max"])


for _s in _sliders:
    _s.observe(on_slider_change, names="value")

slider_box = widgets.VBox(
    [widgets.HTML('<b style="color:var(--gm-on-surface);font-family:var(--gm-font);">'
                  'Nudge the weights and watch the ranking re-sort live:</b>')]
    + _sliders
)
slider_box.add_class("gm-widget-panel")
display(slider_box)


### Compare Mode

In [6]:
def rebuild_compare_controls(top3_df):
    checks = [
        widgets.Checkbox(description=name, value=False, indent=False)
        for name in top3_df["model_name"]
    ]
    compare_btn = widgets.Button(description="Compare Selected")
    compare_result_output = widgets.Output()

    def on_compare_click(_):
        selected = [c.description for c in checks if c.value]
        with compare_result_output:
            clear_output(wait=True)
            if len(selected) < 2:
                display(HTML(
                    '<p style="color:var(--gm-on-surface-variant);font-family:var(--gm-font);'
                    'font-size:13px;">Select at least 2 phones to compare.</p>'
                ))
                return
            comp_df = engine.compare_phones(selected, phones_df)
            header_cols = ["Price", "Camera", "Performance", "Battery", "Display", "Value"]
            header_html = "".join(f"<th>{c}</th>" for c in header_cols)
            body_rows = []
            for model in comp_df.index:
                r = comp_df.loc[model]
                body_rows.append(
                    f"<tr><td>{model}</td><td>₹{r['price_inr']:,}</td>"
                    f"<td>{r['camera_score']:.1f}</td><td>{r['performance_score']:.1f}</td>"
                    f"<td>{r['battery_score']:.1f}</td><td>{r['display_score']:.1f}</td>"
                    f"<td>{r['value_score']:.1f}</td></tr>"
                )
            table_html = (
                '<table class="gm-compare-table"><thead><tr><th>Model</th>'
                f"{header_html}</tr></thead><tbody>{''.join(body_rows)}</tbody></table>"
            )
            display(HTML(f'<div class="gm-wrap">{table_html}</div>'))

    compare_btn.on_click(on_compare_click)
    check_row = widgets.HBox(checks)
    box = widgets.VBox([
        widgets.HTML('<b style="color:var(--gm-on-surface);font-family:var(--gm-font);">'
                     'Select phones to compare side by side:</b>'),
        check_row, compare_btn, compare_result_output,
    ])
    box.add_class("gm-widget-panel")

    with compare_output:
        clear_output(wait=True)
        display(box)


display(compare_output)


Output()

### All-Personas Overview

In [7]:
all_personas_button = widgets.Button(description="Compare All Personas")
all_personas_output = widgets.Output()


def on_all_personas_click(_):
    summary = engine.recommend_all_personas(PERSONAS, phones_df)
    cells_html = []
    for pid, top1 in summary.items():
        pct = min(99, round(top1["match_score"] / 10 * 100))
        cells_html.append(f'''
        <div class="gm-overview-cell">
          <p class="gm-ov-persona">{PERSONAS[pid]["name"]}</p>
          <p class="gm-ov-phone">{top1["model_name"]}</p>
          <p class="gm-ov-pct">{pct}% match</p>
        </div>
        ''')
    html_out = f'<div class="gm-wrap"><div class="gm-overview">{"".join(cells_html)}</div></div>'
    with all_personas_output:
        clear_output(wait=True)
        display(HTML(html_out))


all_personas_button.on_click(on_all_personas_click)
all_personas_box = widgets.VBox([all_personas_button, all_personas_output])
all_personas_box.add_class("gm-widget-panel")
display(all_personas_box)


### Conversational Refine

In [8]:
refine_input = widgets.Text(
    placeholder="e.g. actually I care more about camera",
    layout=widgets.Layout(width="70%"),
)
refine_button = widgets.Button(description="Refine")
refine_output = widgets.Output()


def on_refine_click(_):
    if current_state["weights"] is None:
        with refine_output:
            clear_output(wait=True)
            display(HTML(
                '<p style="color:var(--gm-on-surface-variant);font-family:var(--gm-font);'
                'font-size:13px;">Get a recommendation first, then refine it here.</p>'
            ))
        return
    new_weights = engine.refine_preferences(current_state["weights"], refine_input.value)
    render_results(new_weights, current_state["budget_min"], current_state["budget_max"])
    with refine_output:
        clear_output(wait=True)
        safe_text = html_lib.escape(refine_input.value)
        display(HTML(
            f'<p style="color:var(--gm-on-surface-variant);font-family:var(--gm-font);'
            f'font-size:13px;">Refined based on: “{safe_text}”</p>'
        ))


refine_button.on_click(on_refine_click)
refine_box = widgets.VBox([widgets.HBox([refine_input, refine_button]), refine_output])
refine_box.add_class("gm-widget-panel")
display(refine_box)


---
Run the cells above, then pick a persona (or describe yourself) and click **Find My Galaxy**.

In [9]:
# Initial render so the notebook shows real results immediately on "Run All",
# without requiring a manual click first.
_default_persona = PERSONAS[persona_dropdown.value]
render_results(_default_persona["weights"], _default_persona["budget_min"], _default_persona["budget_max"])
